In [0]:
%pip install google-api-python-client --quiet
dbutils.library.restartPython()

In [0]:
from googleapiclient.discovery import build

api_key = dbutils.secrets.get(scope="youtube-scope", key="youtube-api-key")

youtube = build("youtube", "v3", developerKey=api_key)

In [0]:
channel_id = "UCEGvwrlFT1Ml4VJASkA9rMA"  # Example

response = youtube.channels().list(
    part="contentDetails",
    id=channel_id
).execute()

uploads_playlist_id = response["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]
print("Uploads playlist ID:", uploads_playlist_id)

In [0]:
df = spark.table("workspace_yt_mar.bronze.raw_channels")
df.display()

In [0]:
from pyspark.sql.functions import explode

df_exploded = df.select(explode("items").alias("item"), "ingestion_time")
df_exploded.display()

In [0]:
from pyspark.sql.functions import col

df_flat = df_exploded.select(
    col("item.id").alias("channel_id"),
    col("item.snippet.title").alias("title"),
    col("item.statistics.subscriberCount").alias("subscribers"),
    col("item.statistics.viewCount").alias("views"),
    col("item.statistics.videoCount").alias("videos"),
    col("ingestion_time")
)

df_flat.display()

In [0]:
from pyspark.sql.functions import col

df_clean = df_flat.select(
    col("channel_id"),
    col("title"),
    col("subscribers").cast("long"),
    col("views").cast("long"),
    col("videos").cast("long"),
    col("ingestion_time")
)

df_clean.display()

In [0]:
df_clean = df_clean.fillna({
    "subscribers": 0,
    "views": 0,
    "videos": 0
})

In [0]:
df_clean.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace_yt_mar.silver.channels")

In [0]:
spark.sql("SELECT * FROM workspace_yt_mar.silver.channels").show()

In [0]:
videos = []
nextPageToken = None

while True:
    pl_response = youtube.playlistItems().list(
        part="snippet,contentDetails",
        playlistId=uploads_playlist_id,
        maxResults=50,
        pageToken=nextPageToken
    ).execute()

    videos.extend(pl_response["items"])
    nextPageToken = pl_response.get("nextPageToken")
    if not nextPageToken:
        break

print(f"Fetched {len(videos)} videos")

In [0]:
import json
from pyspark.sql.functions import from_json, schema_of_json, col, lit, current_timestamp

json_rows = [(json.dumps(v),) for v in videos]
df_videos = spark.createDataFrame(json_rows, ["value"])
schema = schema_of_json(lit(json.dumps(videos[0])))
df_videos = df_videos.select(from_json(col("value"), schema).alias("data")).select("data.*")
df_videos = df_videos.withColumn("ingestion_time", current_timestamp())
df_videos.display()

In [0]:
from pyspark.sql.functions import col

df_videos_flat = df_videos.select(
    col("snippet.resourceId.videoId").alias("video_id"),
    col("snippet.title").alias("title"),
    col("snippet.publishedAt").alias("published_at"),
    col("snippet.description").alias("description"),
    col("snippet.channelId").alias("channel_id"),
    col("snippet.channelTitle").alias("channel_title"),
    col("snippet.playlistId").alias("playlist_id"),  # <-- Added playlist info
    col("ingestion_time")
)

df_videos_flat.display()

In [0]:
df_videos_flat.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace_yt_mar.silver.videos")

In [0]:
spark.sql("SELECT * FROM workspace_yt_mar.silver.videos").show(truncate=False)